# **10. Оценка, интерпретация и анализ ошибок моделей (Model Evaluation, Interpretation, and Error Analysis): независимая проверка на тестовой выборке**

* __Цель:__ выполнить независимую оценку заранее зафиксированных моделей-кандидатов на закрытой тестовой выборке, проанализировать характер ошибок прогнозирования и интерпретировать полученные результаты.
* __Задачи:__
  * зафиксировать набор моделей-кандидатов, выбранных по результатам валидационных экспериментов;
  * переобучить выбранные модели на объединенной выборке `train + validation`;
  * выполнить однократную оценку моделей на закрытой test-выборке;
  * сравнить кандидатов по метрикам `MAE`, `RMSE`, `MAPE` и `R²`;
  * подтвердить, что test-выборка не используется для выбора модели, подбора гиперпараметров или изменения набора признаков;
  * проанализировать остатки прогнозирования, ошибки по часам суток и наиболее крупные ошибки прогнозирования;
  * выполнить интерпретацию важности признаков для поддерживаемых индивидуальных моделей;
  * сформулировать итоговые выводы по качеству, устойчивости и ограничениям разработанных моделей.
* __Алгоритм выполнения:__
  1. Загрузить результаты независимой оценки моделей на тестовой выборке.
  2. Проверить состав заранее зафиксированных моделей-кандидатов.
  3. Сопоставить метрики качества на тестовой выборке моделей-кандидатов без повторного ранжирования и выбора модели по test-результатам.
  4. Проанализировать остатки прогнозирования и возможное систематическое смещение прогнозов.
  5. Исследовать распределение ошибок по часам суток.
  6. Выделить и интерпретировать наиболее крупные ошибки прогнозирования.
  7. Проанализировать важность признаков для поддерживаемых индивидуальных моделей.
  8. Отобразить сохраненные визуализации результатов.
  9. Сформулировать аналитические выводы по результатам независимой оценки.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

from traffic_forecasting.config import (
    DATETIME_COLUMN,
    LOCKED_TEST_COMPARISON_PATH,
    LOCKED_TEST_METRICS_PATH,
    LOCKED_TEST_PREDICTIONS_PATH,
    MODEL_EVALUATION_ACTUAL_VS_PREDICTED_FIGURE_PATH,
    MODEL_EVALUATION_CANDIDATES,
    MODEL_EVALUATION_ERROR_BY_HOUR_FIGURE_PATH,
    MODEL_EVALUATION_ERROR_BY_HOUR_PATH,
    MODEL_EVALUATION_FEATURE_IMPORTANCE_FIGURE_PATH,
    MODEL_EVALUATION_FEATURE_IMPORTANCE_PATH,
    MODEL_EVALUATION_LARGE_ERRORS_FIGURE_PATH,
    MODEL_EVALUATION_LARGE_ERRORS_PATH,
    MODEL_EVALUATION_RESIDUAL_COMPARISON_FIGURE_PATH,
    MODEL_EVALUATION_RESIDUAL_SUMMARY_PATH,
    MODEL_EVALUATION_TEST_COMPARISON_FIGURE_PATH,
)
from traffic_forecasting.evaluation import ABSOLUTE_ERROR_COLUMN

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

## **10.1. Фиксация моделей-кандидатов перед независимой оценкой на тестовой выборке (Candidate Selection Before Locked Test Evaluation)**

In [ ]:
candidate_audit = pd.DataFrame(
    {
        "model": MODEL_EVALUATION_CANDIDATES,
        "candidate_role": (
            "strongest individual validation model",
            "best extended ensemble validation model",
        ),
        "selection_basis": (
            "completed validation experiments",
            "completed validation experiments",
        ),
        "training_data_scope": ("train + validation", "train + validation"),
        "evaluation_split": ("locked test", "locked test"),
        "test_used_for_selection": (False, False),
    }
)

display(Markdown("### **Заранее зафиксированные кандидаты для locked test evaluation**"))
display(candidate_audit)

## **10.2. Загрузка результатов независимой оценки (Loading Model Evaluation Artifacts)**

In [ ]:
table_outputs = (
    LOCKED_TEST_METRICS_PATH,
    LOCKED_TEST_COMPARISON_PATH,
    LOCKED_TEST_PREDICTIONS_PATH,
    MODEL_EVALUATION_RESIDUAL_SUMMARY_PATH,
    MODEL_EVALUATION_ERROR_BY_HOUR_PATH,
    MODEL_EVALUATION_LARGE_ERRORS_PATH,
    MODEL_EVALUATION_FEATURE_IMPORTANCE_PATH,
)

figure_outputs = (
    MODEL_EVALUATION_TEST_COMPARISON_FIGURE_PATH,
    MODEL_EVALUATION_ACTUAL_VS_PREDICTED_FIGURE_PATH,
    MODEL_EVALUATION_RESIDUAL_COMPARISON_FIGURE_PATH,
    MODEL_EVALUATION_ERROR_BY_HOUR_FIGURE_PATH,
    MODEL_EVALUATION_LARGE_ERRORS_FIGURE_PATH,
    MODEL_EVALUATION_FEATURE_IMPORTANCE_FIGURE_PATH,
)

required_outputs = table_outputs + figure_outputs
missing_outputs = [path for path in required_outputs if not path.is_file()]

if missing_outputs:
    raise FileNotFoundError(
        "Run 'poetry run python scripts/run_experiments.py --model-evaluation' "
        f"before this notebook. Missing: {missing_outputs}"
    )

test_metrics = pd.read_csv(LOCKED_TEST_METRICS_PATH)
test_comparison = pd.read_csv(LOCKED_TEST_COMPARISON_PATH)
test_predictions = pd.read_csv(
    LOCKED_TEST_PREDICTIONS_PATH,
    parse_dates=[DATETIME_COLUMN],
)
residual_summary = pd.read_csv(MODEL_EVALUATION_RESIDUAL_SUMMARY_PATH)
error_by_hour = pd.read_csv(MODEL_EVALUATION_ERROR_BY_HOUR_PATH)
large_errors = pd.read_csv(
    MODEL_EVALUATION_LARGE_ERRORS_PATH,
    parse_dates=[DATETIME_COLUMN],
)
feature_importance = pd.read_csv(MODEL_EVALUATION_FEATURE_IMPORTANCE_PATH)

loaded_outputs = pd.DataFrame(
    {
        "artifact_group": (["table"] * len(table_outputs) + ["figure"] * len(figure_outputs)),
        "path": [str(path) for path in required_outputs],
        "exists": [path.is_file() for path in required_outputs],
    }
)

display(Markdown("### **Проверка загруженных артефактов независимой оценки моделей**"))
display(loaded_outputs)

## **10.3. Метрики независимой оценки на тестовой выборке (Locked Test Metrics)**

In [ ]:
required_metric_columns = {
    "model",
    "model_group",
    "selection_basis",
    "training_data_scope",
    "split",
    "used_for_model_selection",
    "mae",
    "rmse",
    "mape",
    "r2",
}
missing_metric_columns = sorted(required_metric_columns - set(test_metrics.columns))
if missing_metric_columns:
    raise ValueError(f"Missing locked test metric columns: {missing_metric_columns}")

assert set(test_metrics["split"]) == {"test"}
assert set(test_comparison["split"]) == {"test"}
assert set(test_predictions["split"]) == {"test"}

assert set(test_metrics["training_data_scope"]) == {"train_validation"}
assert set(test_predictions["training_data_scope"]) == {"train_validation"}

assert not test_metrics["used_for_model_selection"].any()
assert not test_comparison["used_for_model_selection"].any()
assert set(test_metrics["selection_basis"]) == {"completed_validation_experiments"}
assert set(test_metrics["model"]) == set(MODEL_EVALUATION_CANDIDATES)
assert "test_rank" not in test_comparison.columns

comparison_columns = [
    "predefined_candidate_order",
    "model",
    "model_group",
    "feature_set_strategy",
    "selection_basis",
    "training_data_scope",
    "mae",
    "rmse",
    "mape",
    "r2",
]
comparison_columns = [column for column in comparison_columns if column in test_comparison.columns]

display(Markdown("### **Метрики качества на закрытой тестовой выборке**"))
display(test_comparison[comparison_columns])

## **10.4. Анализ остатков прогнозирования и систематического смещения (Residual and Bias Analysis)**

In [ ]:
required_residual_columns = {
    "model",
    "split",
    "residual_mean",
    "residual_median",
    "residual_std",
    "mean_absolute_error",
}
missing_residual_columns = sorted(required_residual_columns - set(residual_summary.columns))
if missing_residual_columns:
    raise ValueError(f"Missing residual summary columns: {missing_residual_columns}")

assert set(residual_summary["split"]) == {"test"}

residual_summary_display = residual_summary.sort_values("model").reset_index(drop=True)

display(Markdown("### **Сводная статистика остатков прогнозирования на тестовой выборке**"))
display(residual_summary_display)

## **10.5. Временная структура ошибок и анализ крупных отклонений прогноза (Temporal and Large Error Analysis)**

In [ ]:
required_hourly_columns = {"model", "split", "hour", "mae", "mean_residual"}
missing_hourly_columns = sorted(required_hourly_columns - set(error_by_hour.columns))
if missing_hourly_columns:
    raise ValueError(f"Missing hourly error columns: {missing_hourly_columns}")

required_large_error_columns = {
    "model",
    "split",
    DATETIME_COLUMN,
    ABSOLUTE_ERROR_COLUMN,
}
missing_large_error_columns = sorted(required_large_error_columns - set(large_errors.columns))
if missing_large_error_columns:
    raise ValueError(f"Missing large-error columns: {missing_large_error_columns}")

test_hourly_errors = error_by_hour.loc[error_by_hour["split"] == "test"].copy()
test_large_errors = large_errors.loc[large_errors["split"] == "test"].copy()

if test_hourly_errors.empty:
    raise ValueError("Hourly error analysis requires test rows.")
if test_large_errors.empty:
    raise ValueError("Large-error analysis requires test rows.")

worst_hours = (
    test_hourly_errors.sort_values(["model", "mae"], ascending=[True, False])
    .groupby("model", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

large_error_summary = (
    test_large_errors.groupby("model", as_index=False)
    .agg(
        large_error_count=(ABSOLUTE_ERROR_COLUMN, "size"),
        mean_large_error=(ABSOLUTE_ERROR_COLUMN, "mean"),
        maximum_error=(ABSOLUTE_ERROR_COLUMN, "max"),
    )
    .sort_values("maximum_error", ascending=False)
    .reset_index(drop=True)
)

display(Markdown("### **Пять часов с наибольшим MAE для каждой модели**"))
display(worst_hours[["model", "hour", "mae", "mean_residual"]])

display(Markdown("### **Сводка крупных ошибок на тестовой выборке**"))
display(large_error_summary)

## **10.6. Интерпретация моделей на основе важности признаков (Model Interpretation Through Feature Importance)**

In [ ]:
required_importance_columns = {"model", "feature", "importance"}
missing_importance_columns = sorted(required_importance_columns - set(feature_importance.columns))
if missing_importance_columns:
    raise ValueError(f"Missing feature importance columns: {missing_importance_columns}")

if feature_importance.empty:
    display(
        Markdown(
            "### **Feature importance недоступна для выбранного набора кандидатов**\n\n"
            "В текущем наборе отсутствуют supported individual models, "
            "для которых можно корректно извлечь feature importance."
        )
    )
else:
    feature_importance_display = feature_importance.sort_values(
        ["model", "importance"], ascending=[True, False]
    ).reset_index(drop=True)

    feature_importance_summary = (
        feature_importance_display.groupby("model", as_index=False)
        .agg(
            feature_count=("feature", "size"),
            total_importance=("importance", "sum"),
            maximum_importance=("importance", "max"),
        )
        .sort_values("maximum_importance", ascending=False)
        .reset_index(drop=True)
    )

    top_features_by_model = (
        feature_importance_display.groupby("model", as_index=False).head(15).reset_index(drop=True)
    )

    display(Markdown("### **Сводка feature importance по supported individual models**"))
    display(feature_importance_summary)

    display(Markdown("### **Top-15 наиболее важных преобразованных признаков по каждой модели**"))
    display(top_features_by_model)

## **10.7. Визуализация результатов независимой оценки на тестовой выборке (Result Visualizations)**

In [ ]:
figure_sections = (
    (
        "Сравнение validation-selected candidates на locked test split",
        MODEL_EVALUATION_TEST_COMPARISON_FIGURE_PATH,
    ),
    (
        "Фактические значения и прогнозы моделей-кандидатов",
        MODEL_EVALUATION_ACTUAL_VS_PREDICTED_FIGURE_PATH,
    ),
    (
        "Сравнение распределений residuals",
        MODEL_EVALUATION_RESIDUAL_COMPARISON_FIGURE_PATH,
    ),
    (
        "Средняя абсолютная ошибка по часам суток",
        MODEL_EVALUATION_ERROR_BY_HOUR_FIGURE_PATH,
    ),
    (
        "Крупные ошибки прогнозирования во времени",
        MODEL_EVALUATION_LARGE_ERRORS_FIGURE_PATH,
    ),
    (
        "Feature importance для supported individual models",
        MODEL_EVALUATION_FEATURE_IMPORTANCE_FIGURE_PATH,
    ),
)

figure_audit = pd.DataFrame(
    {
        "description": [description for description, _ in figure_sections],
        "path": [str(path) for _, path in figure_sections],
        "exists": [path.is_file() for _, path in figure_sections],
    }
)

display(Markdown("### **Проверка доступности визуализаций независимой оценки моделей**"))
display(figure_audit)

for description, path in figure_sections:
    display(Markdown(f"### **{description}**"))
    if path.is_file():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Файл визуализации не найден: `{path}`"))

## **10.8. Анализ и интерпретация результатов (Analysis and Interpretation of Model Evaluation Results)**

В рамках этапа оценки, интерпретации и анализа ошибок была выполнена независимая проверка заранее зафиксированных моделей-кандидатов на закрытой test-выборке. Основная задача заключалась в том, чтобы определить, сохраняют ли модели, выбранные по validation-результатам, высокое качество прогнозирования на ранее не использовавшемся test-периоде.

**Ключевые результаты:**
1. **На test-выборке были сопоставлены две заранее зафиксированные модели-кандидата.**
   В независимую оценку были включены `CatBoostRegressor` как сильнейшая индивидуальная модель и `VotingRegressor` как лучший расширенный ансамбль по результатам validation-экспериментов. Обе модели были переобучены на объединенной выборке `train + validation`, после чего один раз оценены на закрытой test-выборке.
2. **По основной test-метрике `RMSE` модель `CatBoostRegressor` показала наилучшее качество.**
   По результатам независимой оценки значение `RMSE` для `CatBoostRegressor` составило `220.604`, а для `VotingRegressor` — `220.965`. Следовательно, по основной метрике, сильнее штрафующей крупные ошибки, меньшую ошибку на test-периоде показала индивидуальная модель `CatBoostRegressor`. При этом по метрике `MAE` значения составили `141.273` для `CatBoostRegressor` и `140.979` для `VotingRegressor`, то есть по средней абсолютной ошибке расширенный ансамбль оказался немного лучше. По `MAPE` также наблюдается небольшое преимущество `VotingRegressor`: `5.778` против `5.902` у `CatBoostRegressor`. Таким образом, test-результаты показывают очень близкое качество двух кандидатов: `CatBoostRegressor` немного лучше по `RMSE`, а `VotingRegressor` — по `MAE` и `MAPE`.
3. **Validation-преимущество ансамбля не полностью перенеслось на test-период.**
   На validation-выборке `VotingRegressor` ранее показывал небольшое преимущество относительно лучшей индивидуальной модели. Однако независимая test-оценка показала, что по основной метрике `RMSE` индивидуальная модель `CatBoostRegressor` оказалась немного устойчивее. Это означает, что выигрыш ансамбля на validation-периоде был умеренным и не полностью сохранился на новом временном интервале. При этом различие между моделями на test-выборке невелико, поэтому обе модели можно рассматривать как практически сопоставимые по качеству.
4. **Test-результаты не используются для повторного выбора модели.**
   Несмотря на различия в test-метриках, данный этап не является новым этапом model selection. Test-выборка использовалась только для независимой оценки качества. Это дополнительно контролируется полями `used_for_model_selection = False` и `training_data_scope = train_validation`. Следовательно, полученные test-результаты следует рассматривать как проверку обобщающей способности ранее выбранных кандидатов, а не как основание для дополнительной настройки модели.
5. **Residual analysis показывает отсутствие выраженного систематического смещения.**
   Среднее значение residuals для `CatBoostRegressor` составило `-5.690`, а для `VotingRegressor` — `-6.159`. Так как residual определяется как `actual - predicted`, отрицательное среднее значение означает, что обе модели в среднем слегка завышают прогноз транспортной нагрузки. При этом абсолютная величина смещения мала по сравнению с масштабом целевой переменной, поэтому выраженного систематического смещения не наблюдается. Более сбалансированное распределение остатков по среднему residual показала модель `CatBoostRegressor`.
6. **Ошибки прогнозирования распределены неравномерно по часам суток.**
   Анализ `MAE` по часам показал, что наибольшие ошибки возникают преимущественно в вечерние и переходные интервалы. Для `CatBoostRegressor` максимальные ошибки наблюдаются в 22:00 (`MAE = 242.282`), 21:00 (`MAE = 227.406`), 07:00 (`MAE = 222.432`), 23:00 (`MAE = 220.970`) и 16:00 (`MAE = 210.084`). Для `VotingRegressor` наиболее сложными оказались 22:00 (`MAE = 248.191`), 21:00 (`MAE = 227.983`), 07:00 (`MAE = 223.435`), 16:00 (`MAE = 213.065`) и 23:00 (`MAE = 212.707`). Это указывает на то, что качество прогноза зависит не только от выбранного алгоритма, но и от временного режима транспортного потока: наиболее сложными являются периоды резкого изменения интенсивности движения.
7. **Крупные ошибки отражают наиболее сложные для прогнозирования наблюдения.**
   Для каждой модели было выделено по `322` наблюдения с крупными ошибками. Средняя крупная ошибка для `CatBoostRegressor` составила `667.090`, а для `VotingRegressor` — `674.614`. Максимальное отклонение для `CatBoostRegressor` составило `2803.703`, а для `VotingRegressor` — `2776.121`. Таким образом, `VotingRegressor` имеет немного меньшую максимальную ошибку, однако по средней величине крупных ошибок `CatBoostRegressor` выглядит немного устойчивее. Это подтверждает, что различия между моделями носят не однозначный, а метрико-зависимый характер.
8. **Feature importance подтверждает решающую роль временных и лаговых признаков.**
   Анализ важности признаков для поддерживаемой индивидуальной модели показал, что наибольший вклад в прогнозирование внесли `traffic_volume_lag_1` (`34.390`), `traffic_volume_lag_168` (`27.333`), `hour_cos` (`14.763`), `traffic_volume_lag_24` (`7.495`) и `hour_sin` (`6.276`). Это хорошо согласуется с природой задачи временного прогнозирования: текущая транспортная нагрузка существенно зависит от ближайшего предыдущего значения, недельной сезонности, суточного цикла и значений за предыдущие сутки.
9. **Визуальный анализ подтверждает результаты табличной оценки.**
   Графики фактических и прогнозных значений показывают, что обе модели в целом воспроизводят динамику транспортной нагрузки на test-периоде. Графики residuals демонстрируют концентрацию ошибок около нуля, что согласуется с небольшими значениями среднего residual. График ошибок по часам суток показывает выраженную зависимость качества прогноза от времени суток, а график крупных ошибок позволяет выделить отдельные участки временного ряда, в которых прогноз становится менее устойчивым.

**Итоговое методологическое резюме:** независимая оценка на test-выборке показала, что выбранные по validation-результатам модели сохраняют работоспособность на ранее не использовавшемся временном интервале. При этом test-результаты следует интерпретировать не как новый этап выбора модели, а как проверку устойчивости ранее принятых решений. `VotingRegressor` показал небольшое преимущество по `MAE` и `MAPE`, однако по основной метрике `RMSE` немного лучше оказался `CatBoostRegressor`. Поэтому с практической точки зрения `CatBoostRegressor` можно рассматривать как более простой и интерпретируемый вариант с сопоставимым или слегка лучшим качеством по `RMSE`, а `VotingRegressor` — как более сложный ансамблевый вариант, который может быть полезен при ориентации на снижение средней абсолютной или относительной ошибки.